# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mariamsherif04/flyrank-ai/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.exists("flyrank-ai"):
        !git clone https://github.com/mariamsherif04/flyrank-ai.git
    os.chdir("flyrank-ai")

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Rebuild the frozen Week-4 baseline exactly, so the comparison is fair
stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

Cloning into 'flyrank-ai'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 158 (delta 63), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 1.87 MiB | 6.49 MiB/s, done.
Resolving deltas: 100% (63/63), done.


## 1. Method choice and why

Method: Logistic Regression first, then Random Forest — my lane's question is "which page should be reviewed first," a ranking problem built on an observed (not future) label. Starting readable (logistic) before going stronger (forest) follows the skill's guidance: simplicity is a feature, complexity only if the comparison earns it.

In [2]:
features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]
print("Features used:", features)
print("Base rate:", y.mean().round(3))

Features used: ['content_age_days', 'days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'word_count']
Base rate: 0.542


## 2. Split design

Split: grouped by client_id, not row-random. A random split would let the same client appear in both train and test, so the model could learn client-specific quirks instead of generalizable signal — the honest question is whether this generalizes to a client it has never seen.

In [3]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df["client_id"]))
print(f"Train: {len(train_idx)} rows, {df['client_id'].iloc[train_idx].nunique()} clients")
print(f"Test:  {len(test_idx)} rows, {df['client_id'].iloc[test_idx].nunique()} clients")
print("Overlap in client_id between train/test (should be 0):",
      len(set(df['client_id'].iloc[train_idx]) & set(df['client_id'].iloc[test_idx])))

Train: 22885 rows, 24 clients
Test:  7115 rows, 8 clients
Overlap in client_id between train/test (should be 0): 0


## 3. Train + compare vs my baseline

Baseline vs. Logistic Regression vs. Random Forest, same test split, same metric (precision@50), same base rate for reference.

In [4]:
Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
hand_rule_test = df["hand_rule_score"].iloc[test_idx].values

logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).fit(Xtr, ytr)
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced").fit(Xtr, ytr)

results = pd.DataFrame({
    "method": ["Baseline rule", "Logistic Regression", "Random Forest"],
    "precision_at_50": [
        precision_at_k(hand_rule_test, yte.values, 50),
        precision_at_k(logreg.predict_proba(Xte)[:,1], yte.values, 50),
        precision_at_k(rf.predict_proba(Xte)[:,1], yte.values, 50),
    ],
    "base_rate": [yte.mean()]*3,
})
print(results.round(3))

                method  precision_at_50  base_rate
0        Baseline rule             0.62      0.517
1  Logistic Regression             0.66      0.517
2        Random Forest             0.68      0.517


## 4. Errors and interpretation

Error analysis: feature importances from the Random Forest, then 3 concrete wrong cases inspected by hand.

In [5]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Feature importances:")
print(importances.round(3))
print("\nTop feature makes sense if it's a signal already confirmed in the Week-4 signal check")
print("(staleness/impressions) rather than something suspiciously perfect — a red flag for leakage.")

test_df = df.iloc[test_idx].copy()
test_df["rf_score"] = rf.predict_proba(Xte)[:,1]
wrong = test_df.sort_values("rf_score", ascending=False).head(50)
wrong = wrong[wrong["is_declining_label"] == 0].head(3)
print("\n3 concrete wrong cases (flagged high, not actually declining):")
print(wrong[["content_id","rf_score","days_since_last_update","impressions_90d","avg_position"]])

Feature importances:
impressions_90d           0.275
avg_position              0.249
content_age_days          0.161
word_count                0.156
ctr                       0.121
days_since_last_update    0.037
dtype: float64

Top feature makes sense if it's a signal already confirmed in the Week-4 signal check
(staleness/impressions) rather than something suspiciously perfect — a red flag for leakage.

3 concrete wrong cases (flagged high, not actually declining):
                 content_id  rf_score  days_since_last_update  \
2357   content_8f1409b2674e     0.990                     104   
22526  content_1d0963b56227     0.985                     104   
4480   content_f5b35bbb42fc     0.980                      13   

       impressions_90d  avg_position  
2357               209          20.0  
22526             3445          39.0  
4480              3171          31.8  


In [6]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, Xte, yte, n_repeats=10, random_state=42, scoring="average_precision")
perm_importances = pd.Series(perm.importances_mean, index=features).sort_values(ascending=False)
print("Permutation importance (drop in score when each feature is shuffled):")
print(perm_importances.round(4))
print("\nUnlike built-in feature_importances_, this measures the ACTUAL performance drop")
print("when a feature is shuffled — a more honest signal of what the model relies on.")

Permutation importance (drop in score when each feature is shuffled):
impressions_90d           0.0598
avg_position              0.0236
content_age_days          0.0205
ctr                       0.0133
word_count               -0.0086
days_since_last_update   -0.0112
dtype: float64

Unlike built-in feature_importances_, this measures the ACTUAL performance drop
when a feature is shuffled — a more honest signal of what the model relies on.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.